# Fine-tuning con LoRA: modelo original vs modelo + adaptador

**Enunciado:** ajustar con LoRA un modelo pequeño sobre un dataset mínimo para que adopte un estilo o formato concreto y comparar el modelo base con el ajustado ante la misma pregunta.

1. Preparar ejemplos pregunta-respuesta con el estilo objetivo.
2. Entrenar el adaptador LoRA.
3. Probar base vs ajustado.
4. Comentar la diferencia.

**Cómo se resuelve aquí:** el entrenamiento ya se hizo en local con `01_tunning_with_lora.py` (modelo `Qwen/Qwen3-0.6B`). Este notebook **no entrena**: carga el modelo original y el modelo + adaptador (igual que `02_inference.py`) y los compara.

> Ejecutar con el directorio de trabajo en la carpeta del proyecto (donde están `utils.py` y `results/`).

In [9]:
import json
import shutil
from pathlib import Path

from tabulate import tabulate
from transformers import AutoTokenizer, PreTrainedTokenizerBase

from colorize import colorize
from utils import get_dataset, load_model

OPEN_MODEL_ID = 'Qwen/Qwen3-0.6B'
ADAPTER_FOLDER = './results/lora/adapters/20261108_003826'  # mismo adaptador que 02_inference.py
TERMINAL_WIDTH: int = shutil.get_terminal_size().columns

## Paso 1 · Dataset con el estilo objetivo

El dataset (`assets/datasets/dataset.json`) contiene pares `prompt` → `completion` sobre la academia *Nube*, divididos en `train`, `validation` y `test`.

In [10]:
datasets = get_dataset()

print(f'train: {len(datasets.train)} | validation: {len(datasets.validation)} | test: {len(datasets.test)}')

rows = [[ex['prompt'], ex['completion']] for ex in datasets.train.select(range(5))]
print(tabulate(rows, headers=['Prompt', 'Completion'], tablefmt='rounded_grid', maxcolwidths=[45, 60]))

train: 70 | validation: 20 | test: 10
╭──────────────────────────────────────────────┬──────────────────────────────────────────────────────────────╮
│ Prompt                                       │ Completion                                                   │
├──────────────────────────────────────────────┼──────────────────────────────────────────────────────────────┤
│ ¿Cuál es el horario de atención de Nube?     │ La academia Nube atiende de lunes a viernes de 9:00 a 18:00. │
├──────────────────────────────────────────────┼──────────────────────────────────────────────────────────────┤
│ ¿A qué horas puedo contactar con la academia │ La academia Nube atiende de lunes a viernes de 9:00 a 18:00. │
│ entre semana?                                │                                                              │
├──────────────────────────────────────────────┼──────────────────────────────────────────────────────────────┤
│ ¿Cuándo abre y cierra Nube de lunes a        │ La academia Nube 

## Paso 2 · Adaptador LoRA entrenado

El adaptador se entrenó con `01_tunning_with_lora.py`. Aquí solo se muestra su configuración.

In [11]:
adapter_config = json.loads((Path(ADAPTER_FOLDER) / 'adapter_config.json').read_text(encoding='utf-8'))

print(tabulate(
    [
        ['Modelo base', adapter_config['base_model_name_or_path']],
        ['r', adapter_config['r']],
        ['lora_alpha', adapter_config['lora_alpha']],
        ['target_modules', ', '.join(sorted(adapter_config['target_modules']))],
        ['Carpeta del adaptador', ADAPTER_FOLDER],
    ],
    tablefmt='rounded_grid',
))

╭───────────────────────┬─────────────────────────────────────────╮
│ Modelo base           │ Qwen/Qwen3-0.6B                         │
├───────────────────────┼─────────────────────────────────────────┤
│ r                     │ 16                                      │
├───────────────────────┼─────────────────────────────────────────┤
│ lora_alpha            │ 32                                      │
├───────────────────────┼─────────────────────────────────────────┤
│ target_modules        │ k_proj, o_proj, q_proj, v_proj          │
├───────────────────────┼─────────────────────────────────────────┤
│ Carpeta del adaptador │ ./results/lora/adapters/20261108_003826 │
╰───────────────────────┴─────────────────────────────────────────╯


## Paso 3 · Cargar modelo original y modelo + adaptador

In [12]:
tokenizer: PreTrainedTokenizerBase = AutoTokenizer.from_pretrained(OPEN_MODEL_ID)

original_model = load_model()
trained_model = load_model(ADAPTER_FOLDER)

In [13]:
def ask(model, question: str, max_new_tokens: int = 100) -> str:
    """Genera la respuesta del modelo (determinista) para una pregunta."""

    inputs = tokenizer.apply_chat_template(
        [{'role': 'user', 'content': question}],
        tokenize=True,
        add_generation_prompt=True,
        enable_thinking=False,
        return_tensors='pt',
        return_dict=True,
    ).to(model.device)

    outputs = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=False,
    )

    new_tokens = outputs[0][inputs['input_ids'].shape[1]:]

    return tokenizer.decode(new_tokens, skip_special_tokens=True)

### 3.1 · La misma pregunta, dos modelos

In [14]:
question = '¿Cuál es el coste total de inscribirse en el curso básico?'

print(colorize('Pregunta: ', 'purple') + question)
print(colorize('Respuesta del modelo original: ', 'purple') + ask(original_model, question))
print(colorize('Respuesta del modelo+adaptador: ', 'purple') + ask(trained_model, question))

Pregunta: ¿Cuál es el coste total de inscribirse en el curso básico?
Respuesta del modelo original: El coste total de inscribirse en el curso básico depende de la institución, el tipo de curso y las condiciones de inscripción. En general, en muchos países, el costo inicial puede ser de **$100 a $200** por curso. Sin embargo, es importante consultar directamente con la institución para obtener un informe detallado.
Respuesta del modelo+adaptador: El coste total de inscribirse en el curso básico es **199 euros**.


### 3.2 · Batería de preguntas (las mismas de `02_inference.py`)

In [15]:
questions = [
    '¿Puedo llamar a la academia un lunes a las 8:30?',
    '¿Nube abre algún sábado por la mañana?',
    '¿Cuál es el coste total de inscribirse en el curso básico?',
    '¿Aceptan pago con tarjeta de crédito para el curso básico?',
    '¿Tengo clase del curso básico los miércoles?',
    '¿Cuántos minutos tengo que reservar para cada sesión del curso?',
    '¿Las clases se dan de forma presencial o a través de alguna plataforma?',
    '¿Puedo descargar el material para leerlo sin conexión? ¿En qué formato viene?',
    '¿Podré ver la grabación de una clase dentro de dos meses?',
    '¿A qué email escribo si CampusNube no me carga el vídeo de la clase?',
    'Envié una consulta a soporte ayer, ¿cuándo debería tener respuesta como muy tarde?',
    'Si falto a 3 de cada 10 clases, ¿puedo obtener el certificado?',
    '¿Tengo que pedir que me envíen el certificado por correo postal?',
    'Quiero darme de baja el próximo lunes, ¿a qué correo escribo y con cuánto margen?',
    'No me acuerdo de mi clave de CampusNube, ¿qué tengo que hacer para entrar?',
]

responses = {'original_model': [], 'trained_model': []}

for key, model in (('original_model', original_model), ('trained_model', trained_model)):
    for q in questions:
        responses[key].append(ask(model, q))

In [16]:
for i, q in enumerate(questions):
    print(colorize('Pregunta: ', 'purple') + q)
    print(colorize('Respuesta del modelo original: ', 'purple') + responses['original_model'][i])
    print(colorize('Respuesta del modelo+adaptador: ', 'purple') + responses['trained_model'][i])
    print()
    print('-' * TERMINAL_WIDTH)
    print()

Pregunta: ¿Puedo llamar a la academia un lunes a las 8:30?
Respuesta del modelo original: No, no puedes llamar a la academia un lunes a las 8:30. La academia suele estar abierta el día completo, incluyendo lunes, y suele tener horarios regulares. Si necesitas ayuda en alguna actividad específica, por favor comunica conmigo.
Respuesta del modelo+adaptador: Sí, es posible llamar a la academia el lunes a las 8:30.

--------------------------------------------------------------------------------

Pregunta: ¿Nube abre algún sábado por la mañana?
Respuesta del modelo original: No, no abre el sábio por la mañana. ¿Quieres que te ayude con algo?
Respuesta del modelo+adaptador: La pregunta es en español. La nube (como Amazon Web Services) no tiene horarios definidos por día. Si tienes alguna pregunta, ¡avísame!

--------------------------------------------------------------------------------

Pregunta: ¿Cuál es el coste total de inscribirse en el curso básico?
Respuesta del modelo original: El 

## Paso 4 · Comentario de la diferencia

Para valorar las respuestas se han contrastado con las respuestas reales del dataset (`assets/datasets/dataset.json`): lunes a viernes de 9:00 a 18:00, sábados cerrado, curso básico a 60 euros, pago por transferencia, clases martes y jueves de 60 minutos en CampusNube, material en PDF, grabaciones 30 días, soporte `ayuda@nube.example` (48 h), certificado con el 80 % de asistencia enviado por email y baja a `baja@nube.example` con 7 días de antelación.

**Modelo original**
- Respuestas largas y genéricas, con listas y negritas, que a menudo terminan ofreciendo ayuda ("¡avísame!").
- No conoce la academia *Nube*: inventa datos plausibles (p. ej. "**$100 a $200**" de coste, plataformas como "PESE" o "Sesiones Online") o contesta con consejos de carácter general.
- En algunas preguntas acierta por casualidad (p. ej. que no se puede llamar un lunes a las 8:30), pero con una justificación incorrecta.

**Modelo + adaptador**
- **Cambia claramente el estilo**: respuestas más cortas y directas, en una sola frase, con el tono del dataset (p. ej. "El coste total de inscribirse en el curso básico es **199 euros**").
- **Aciertos**: el formato del material (PDF), que no se acepta tarjeta de crédito (el pago es por transferencia) y que con el 70 % de asistencia no se obtiene el certificado.
- **Errores**: el estilo se aprende mejor que los hechos. Responde con seguridad datos incorrectos (199 € en vez de 60 €, Udemy en vez de CampusNube, "sí" a llamar el lunes a las 8:30, clase los miércoles), confunde *Nube* con la nube de computación en la pregunta del sábado y no recupera los correos ni los plazos (`ayuda@`, `baja@`, 48 h, 7 días). En las preguntas más largas (vídeo que no carga, contraseña) cae en respuestas genéricas parecidas a las del modelo original.

**Diferencia principal**
El adaptador logra que el modelo adopte el formato breve y asertivo del dataset, pero no consigue que memorice de forma fiable los datos concretos de la academia. Además, esa asertividad hace que los errores sean menos evidentes que en el modelo original, que al menos es más cauteloso ("depende de la institución").

**Limitaciones y posibles mejoras**
- Modelo muy pequeño (0,6B parámetros) y dataset de solo 70 ejemplos de entrenamiento, con muchas paráfrasis de la misma respuesta.
- La métrica elegida (loss / accuracy a nivel de token) no mide si el dato es correcto; convendría evaluar con las respuestas esperadas del split `test`.
- Mejoras posibles: más ejemplos y más variados, más épocas o mayor rango, un modelo base mayor y, si lo que se necesita es que consulte datos reales, usar RAG en lugar de fine-tuning.